In [1]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

from psycopg.rows import dict_row
from psycopg_pool import AsyncConnectionPool
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from app.core.config import settings
from dotenv import load_dotenv

load_dotenv()

# 创建连接池
checkpoint_pool = AsyncConnectionPool(
    conninfo=settings.db.checkpoint_url,
    min_size=1,  # 池最小连接数
    max_size=5,  # 池最大连接数
    kwargs={
        "autocommit": True,  # 自动提交事务
        "prepare_threshold": 0,  # 不做预准备sql
        "row_factory": dict_row,  # 查询结果以dict返回
    },
    open=False,  # 不自动创建连接
)
# 初始化连接
await checkpoint_pool.open()
# 等待连接池就绪
await checkpoint_pool.wait()

# 初始化checkpointer
checkpointer = AsyncPostgresSaver(checkpoint_pool)
# 自动建表（与checkpointer存储有关的表）
await checkpointer.setup()

In [2]:
from langchain_core.runnables import RunnableConfig
from langchain.messages import HumanMessage
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# 1.准备和加载环境变量
load_dotenv()

# 2.初始化模型
model = init_chat_model(
        "deepseek-v4-flash",
        extra_body={"thinking": {"type": "disabled"}}
    )

# 3.准备tools
tools = []

# 4.创建Agent，绑定model、tools、checkpointer
agent = create_agent(
    model=model,
    tools=tools, # 可省略
    checkpointer=checkpointer,
)

# 5.调用
# config = {"configurable": {"thread_id": "t1"}}
config = RunnableConfig(configurable={"thread_id": "t1"})


# 第一次调用，告知AI我的信息
response = await agent.ainvoke(
    {"messages": [HumanMessage(content="你好，我叫虎哥，我最喜欢猫猫。")]},
    config # 调用时添加thread_id，区分不同会话
)

# print(response['messages'][-1]['content'])
print(response['messages'][-1].content)

print('*' * 100)

# 第二次调用，询问我的信息，这次带上thread_id，唤起记忆
response = await agent.ainvoke(
    {"messages": [HumanMessage(content="我最喜欢的动物是什么？")]},
    config # 调用时添加thread_id
)

print(response['messages'][-1].content)

你好呀，虎哥！🐾 看到你对猫猫的喜欢，感觉屏幕前都冒出了毛茸茸的幸福感～（脑补一只猫猫探头.gif）  
既然这么爱猫，要不要来聊点猫猫玄学？比如——  
*猫主子为什么总在你工作时精准踩键盘？*  
*还是说，你其实偷偷练成了“猫语十级”？* 😼  
（附赠一个云撸猫预警：本AI已加载🍤小鱼干投喂、🐈呼噜声模拟等隐藏技能，随时待机！）
****************************************************************************************************
哈哈，虎哥你这是在考我呀！既然你叫“虎哥”，还说过最爱猫猫——那我猜：  
**你最喜欢的动物必须是——猫！**（毕竟“虎”也是大猫嘛🐯，猫科全家桶都是你的心头好对不对？）  

不过嘛……万一你偷偷变心喜欢上水豚或企鹅了？  
（试探性递出小鱼干🍤）——要不要给个提示？我保证不告诉猫主子！ 😼


In [6]:
snapshot = await agent.aget_state(config)
messages = snapshot.values.get("messages", [])

for message in messages:
    message.pretty_print()
    print(22,message.content)

================================ Human Message =================================

你好，我叫虎哥，我最喜欢猫猫。
22 你好，我叫虎哥，我最喜欢猫猫。
================================== Ai Message ==================================

你好呀，虎哥！🐾 看到你对猫猫的喜欢，感觉屏幕前都冒出了毛茸茸的幸福感～（脑补一只猫猫探头.gif）  
既然这么爱猫，要不要来聊点猫猫玄学？比如——  
*猫主子为什么总在你工作时精准踩键盘？*  
*还是说，你其实偷偷练成了“猫语十级”？* 😼  
（附赠一个云撸猫预警：本AI已加载🍤小鱼干投喂、🐈呼噜声模拟等隐藏技能，随时待机！）
22 你好呀，虎哥！🐾 看到你对猫猫的喜欢，感觉屏幕前都冒出了毛茸茸的幸福感～（脑补一只猫猫探头.gif）  
既然这么爱猫，要不要来聊点猫猫玄学？比如——  
*猫主子为什么总在你工作时精准踩键盘？*  
*还是说，你其实偷偷练成了“猫语十级”？* 😼  
（附赠一个云撸猫预警：本AI已加载🍤小鱼干投喂、🐈呼噜声模拟等隐藏技能，随时待机！）
================================ Human Message =================================

我最喜欢的动物是什么？
22 我最喜欢的动物是什么？
================================== Ai Message ==================================

哈哈，虎哥你这是在考我呀！既然你叫“虎哥”，还说过最爱猫猫——那我猜：  
**你最喜欢的动物必须是——猫！**（毕竟“虎”也是大猫嘛🐯，猫科全家桶都是你的心头好对不对？）  

不过嘛……万一你偷偷变心喜欢上水豚或企鹅了？  
（试探性递出小鱼干🍤）——要不要给个提示？我保证不告诉猫主子！ 😼
22 哈哈，虎哥你这是在考我呀！既然你叫“虎哥”，还说过最爱猫猫——那我猜：  
**你最喜欢的动物必须是——猫！**（毕竟“虎”也是大猫嘛🐯，猫科全家桶都是你的心头好对不对？）  

不过嘛……万一你偷偷变心喜欢上水豚或企鹅了？  


In [4]:
# thread_id = config["configurable"]["thread_id"]
# await checkpointer.adelete_thread(thread_id)
# print('*' * 50)

In [5]:
# snapshot = await agent.aget_state(config)
# messages = snapshot.values.get("messages", [])
#
# for message in messages:
#     message.pretty_print()